In [16]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [2]:
dataset = pd.read_csv("data_treated.csv")

In [3]:
categorical_variables = []
with open("categorical.txt", "r") as file:
    for line in file:
        categorical_variables.append(line.strip())

numerical_variables = []
with open("numerical.txt", "r") as file:
    for line in file:
        numerical_variables.append(line.strip())

In [4]:
survival_dataset = dataset.drop('Unnamed: 0', axis=1)

In [5]:
median_imputer(survival_dataset, numerical_variables)

'Done'

In [6]:
feature_columns = categorical_variables + numerical_variables

In [7]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
# Since every duplicated patient has a consistent event indicator (regional ln recurrence) I can use this method
groups = survival_dataset['PERSON_ID']
event_indicator = survival_dataset['Regional LN recurrence']

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 500],  # 100 can already be stable; 500 is often enough
    "min_samples_split": [2, 5],      # Prevent overfitting; 10+ rarely helps in clinical data
    "min_samples_leaf": [1, 5, 10],   # Key for balancing bias-variance
    "max_features": ["sqrt", 0.5],    # sqrt is typical; 0.5 can boost diversity
    "max_depth": [None, 10],          # None (unlimited) vs limited depth
    "bootstrap": [True]               # Keep True — RSF depends on bootstrapping
}

# {'bootstrap': True, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 200}
# Best C-index: 0.7380676257590892

sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=event_indicator, groups=groups)
rsf = RandomSurvivalForest(n_jobs=-1, random_state=12)

In [10]:
grid_search = GridSearchCV(
    estimator=rsf,
    param_grid=param_grid,
    cv=cv,
    scoring=score_survival_model,
    n_jobs=-1,
    verbose=1
)

In [11]:
grid_search.fit(data_x, data_y)

Fitting 4 folds for each of 72 candidates, totalling 288 fits


GridSearchCV(cv=<generator object _BaseKFold.split at 0x000001F6A044FD00>,
             estimator=RandomSurvivalForest(n_jobs=-1, random_state=12),
             n_jobs=-1,
             param_grid={'bootstrap': [True], 'max_depth': [None, 10],
                         'max_features': ['sqrt', 0.5],
                         'min_samples_leaf': [1, 5, 10],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200, 500]},
             scoring=<function score_survival_model at 0x000001F6A04EF880>,
             verbose=1)

In [13]:
best_parameters = grid_search.best_params_
best_c_index = grid_search.best_score_
best_model = grid_search.best_estimator_
print("Best parameters:", best_parameters)
print("Best C-index:", best_c_index)

Best parameters: {'bootstrap': True, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 200}
Best C-index: 0.7380676257590892


In [ ]:
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=12)
'''rsf = RandomSurvivalForest(
    n_estimators=200, min_samples_split=5, min_samples_leaf=1, max_features=0.5, max_depth=10, bootstrap= True, n_jobs=-1, random_state=13
)'''

In [21]:
results = train_evaluate(data_x, data_y, groups, sgkf, best_model)
results

,fold,c_index,brier_percentile,brier_alltimes,brier_fixed
0,0,0.752872,0.013473,0.035143,0.026554
1,1,0.720025,0.011445,0.027520,0.023824
2,2,0.724766,0.014343,0.058589,0.017434
3,3,0.754608,0.004499,0.156422,0.009891
4,mean,0.738068,0.010940,0.069419,0.019426
5,std,0.018214,0.004462,0.059490,0.007417
